In [ ]:
%run ../packages.py



In [28]:
races_2024 = ['bahrain',
              'saudi-arabian',
              'australian',
              'japanese',
              'chinese',
              'miami',
              'emilia-romagna',
              'monaco',
              'spanish',
              'canadian',
              'austrian',
              'british',
              'hungarian',
              'belgian',
              'dutch',
              'italian',
              'azerbaijan',
              'singapore',
              'united-states',
              'mexican',
              'sao-paulo',
              'las-vegas',
              'qatar',
              'abu-dhabi']

In [29]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


def fetch_race_data(year, race_name):
    # Construct the URL
    url = f"https://pitwall.app/races/{year}-{race_name}-grand-prix"
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to retrieve data for {race_name} {year}")
        return None

    # Parse the page content
    soup = BeautifulSoup(response.text, 'html.parser')

    # Extract Race Results
    race_results = []
    race_table = soup.find('table', {'id': 'data-table'})
    if race_table:
        headers = [th.text.strip()
                   for th in race_table.find('thead').find_all('th')]
        for row in race_table.find('tbody').find_all('tr'):
            cells = [cell.text.strip() for cell in row.find_all('td')]
            race_results.append(dict(zip(headers, cells)))

    # Extract Qualifying Results
    qualifying_results = []
    qualifying_table = soup.find('table', {'id': 'qualifying-results'})
    if qualifying_table:
        headers = [th.text.strip()
                   for th in qualifying_table.find('thead').find_all('th')]
        for row in qualifying_table.find('tbody').find_all('tr'):
            cells = [cell.text.strip() for cell in row.find_all('td')]
            qualifying_results.append(dict(zip(headers, cells)))

    return {
        'race_results': race_results,
        'qualifying_results': qualifying_results
    }


def parse_data_tables(year, race_name):
    url = f"https://pitwall.app/races/{year}-{race_name}-grand-prix"
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to retrieve data for {race_name} {year}")
        return None
    soup = BeautifulSoup(response.text, 'html.parser')

    tables = soup.find_all('table', class_='data-table')
    print(f"Found {len(tables)} tables.")

    dataframes = []

    for i, table in enumerate(tables):
        headers = [th.get_text(strip=True) for th in table.find_all('th')]
        rows = []

        for row in table.find_all('tr')[1:]:  # Skip header row
            cells = [td.get_text(strip=True) for td in row.find_all('td')]
            if cells:
                rows.append(cells)

        df = pd.DataFrame(rows, columns=headers if headers else None)
        dataframes.append(df)

    return dataframes


def save_race(year, slug, url, tables, directory):
    race_data = {
        "year": year,
        "slug": slug,
        "url": url,
        "tables": tables
    }

    # os.makedirs("race_data", exist_ok=True)
    with open(f"{directory}/{year}_{slug}.pkl", "wb") as f:
        pickle.dump(race_data, f)
    return


raw_path = '/Users/bradkittrell/Projects/F1_forecast/F1_forecast/Data/raw'

# remove the file extension
existing_races_saved_2024 = [i.split('/')[-1].split('_')[1].split('.')[0]
                             for i in glob.glob(f"{raw_path}/*.pkl")]

existing_races_saved_2024

['japanese',
 'chinese',
 'austrian',
 'bitish',
 'miami',
 'hungarian',
 'mexican',
 'monaco',
 'azerbaijan',
 'qatar',
 'australian',
 'singapore',
 'dutch',
 'belgian',
 'abu-dhabi',
 'british',
 'spanish',
 'italian',
 'sigapore',
 'bahrain',
 'united-states',
 'saudi-arabian',
 'las-vegas',
 'emilia-romagna',
 'sao-paulo']

In [30]:
import time


for race in enumerate(races_2024):
    if not race[1] in existing_races_saved_2024:

        print(f"Fetching data for {race[1]}")
        race_url = f"https://pitwall.app/races/2024-{race[1]}-grand-prix"

        dfs = parse_data_tables(2024, race[1])

        save_race(2024, race[1], race_url, dfs, raw_path)
        time.sleep(5)

Fetching data for canadian
Found 11 tables.


In [3]:
url = "https://pitwall.app/races/2024-abu-dhabi-grand-prix"

dfs = parse_data_tables(2024, 'abu-dhabi')

Found 11 tables.


In [11]:
with open("/Users/bradkittrell/Projects/F1_forecast/F1_forecast/Data/raw/2024_abu-dhabi.pkl", "rb") as f:
    race_data = pickle.load(f)

# Check what you got
print(f"Year: {race_data['year']}")
print(f"Slug: {race_data['slug']}")
print(f"URL: {race_data['url']}")

# The list of DataFrames
tables = race_data['tables']
for i, df in enumerate(tables):
    print(f"\nTable {i+1}:")
    print(df.shape)

Year: 2024
Slug: abu-dhabi
URL: https://pitwall.app/races/2024-abu-dhabi-grand-prix

Table 1:
(20, 7)

Table 2:
(20, 8)

Table 3:
(20, 6)

Table 4:
(20, 6)

Table 5:
(20, 6)

Table 6:
(19, 6)

Table 7:
(5, 4)

Table 8:
(5, 3)

Table 9:
(24, 7)

Table 10:
(10, 7)

Table 11:
(16, 6)


In [6]:
def save_race(year, slug, url, tables, directory):
    race_data = {
        "year": year,
        "slug": slug,
        "url": url,
        "tables": tables
    }

    # os.makedirs("race_data", exist_ok=True)
    with open(f"{directory}/{year}_{slug}.pkl", "wb") as f:
        pickle.dump(race_data, f)
    return


raw_path = '/Users/bradkittrell/Projects/F1_forecast/F1_forecast/Data/raw'

save_race(2024, 'abu-dhabi', url, dfs, raw_path)